In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

sales_long = pd.read_parquet(PROCESSED_DIR / "sales_long_clean.parquet")
print(sales_long.shape)  

(46027957, 20)


Percentage of Zeros After Trimming

In [2]:
zero_pct_clean = (sales_long['sold'] == 0).mean()
print(f"Zero-sale rows (post-trim): {zero_pct_clean:.1%}")

Zero-sale rows (post-trim): 59.7%


Time-Based Train/Validation Split (Last 28 Days = Validation)

In [3]:
max_date = sales_long['date'].max()
val_start = max_date - pd.Timedelta(days=27)

train = sales_long[sales_long['date'] < val_start].copy()
val = sales_long[sales_long['date'] >= val_start].copy()

print(f"Train: {train['date'].min().date()} → {train['date'].max().date()}  ({len(train):,} rows)")
print(f"Val:   {val['date'].min().date()} → {val['date'].max().date()}  ({len(val):,} rows)")

Train: 2011-01-29 → 2016-03-27  (45,174,237 rows)
Val:   2016-03-28 → 2016-04-24  (853,720 rows)


Naive baseline

In [4]:
last_value = train.sort_values('date').groupby('id', observed=True)['sold'].last()

naive_pred = val[['id', 'date', 'sold']].copy()
naive_pred['pred'] = naive_pred['id'].map(last_value)
naive_pred['pred'] = naive_pred['pred'].fillna(0)

naive_pred.head()

,id,date,sold,pred
57473650,HOBBIES_1_001_CA_1_validation,2016-03-28,1,1
57473651,HOBBIES_1_002_CA_1_validation,2016-03-28,1,1
57473652,HOBBIES_1_003_CA_1_validation,2016-03-28,0,0
57473653,HOBBIES_1_004_CA_1_validation,2016-03-28,0,6
57473654,HOBBIES_1_005_CA_1_validation,2016-03-28,1,0


Seasonal Naive

In [5]:
last_days = train[train['date'] > train['date'].max() - pd.Timedelta(days=7)][['id', 'date', 'sold']].copy()
last_days['dow'] = last_days['date'].dt.dayofweek
seasonal_map = last_days.set_index(['id', 'dow'])['sold']

val['dow'] = val['date'].dt.dayofweek
val['pred_snaive'] = val.set_index(['id', 'dow']).index.map(seasonal_map)
val['pred_snaive'] = val['pred_snaive'].fillna(0)

Baseline Evaluation (RMSE/MAE) for Initial Comparison

In [6]:
import numpy as np

rmse_naive = np.sqrt(((naive_pred['sold'] - naive_pred['pred'])**2).mean())
mae_naive = (naive_pred['sold'] - naive_pred['pred']).abs().mean()
print(f"Naive          → RMSE={rmse_naive:.3f}  MAE={mae_naive:.3f}")

rmse_snaive = np.sqrt(((val['sold'] - val['pred_snaive'])**2).mean())
mae_snaive = (val['sold'] - val['pred_snaive']).abs().mean()
print(f"Seasonal Naive → RMSE={rmse_snaive:.3f}  MAE={mae_snaive:.3f}")

Naive          → RMSE=3.301  MAE=1.350
Seasonal Naive → RMSE=2.759  MAE=1.232


Baseline Evaluation with WRMSSE (src/metrics.py)

Scale: mean squared one-step difference over train, from each series' first non-zero sale.
Weights: dollar sales over the last 28 days of train (sold x sell_price), across all 12 M5 hierarchy levels.

In [ ]:
import sys
sys.path.insert(0, "..")

from src.data import load_sell_prices, to_wide
from src.metrics import WRMSSEEvaluator
from src.models.baselines import naive, seasonal_naive

evaluator = WRMSSEEvaluator.from_long(train, load_sell_prices())
train_wide = to_wide(train)
actual = to_wide(val, dates=pd.date_range(val['date'].min(), periods=28))

level_scores = pd.DataFrame({
    "Naive": evaluator.level_scores(actual, naive(train_wide)),
    "Seasonal Naive": evaluator.level_scores(actual, seasonal_naive(train_wide)),
})
print(level_scores.mean().round(4).to_string())
level_scores.round(4)